# NLP

# Setup 
- **NLTK** - classic teaching toolkit (tokenizers, stemmers, corpora)
- **scikit-learn** - vectorizes + ML models
- **gensim** = Word2Vec training
- **spaCy** - industrial NER & dependency parsing

In [1]:
import sys
import subprocess
import nltk

packages = [
    'punkt',                # Tokenizers
    'stopwords',            # stopword lists
    'wordnet', 'omw-1.4',   # lemmatizer dictionary
    'averaged_perceptron_tagger', # POS tagger
    'averaged_perceptron_tagger_eng',
    'maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words', # NER chunker
    'vader_lexicon',        # sentiment lexicon
]

for pkg in packages:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f'Could not download {pkg} : {e}')

print('setup complete')

setup complete


# 1. Tokenization

**Idea:** computers can't read a sentence as one blob - we break it into pieces called tokens (usually words, cometimes sentences or sub-words). Tokenization is always the first step of any NLP pipeline.

**Why it matters:** Every later step (counting words, taggin, classifying) operates on tokens. Bad tokenization = bad everything downstream

**Watch out for:** punctuation, contractions(`don't` -> `do` + `n't`) and abbreviations(U.S.A.)


In [2]:
from nltk.tokenize import word_tokenize, sent_tokenize

text = "Dr. Smith isn't happy. NLP is fun, and it's powerful! Visit openai.com today."

# Sentence tokenization - splits on sentece boundaries (note: it keeps 'Dr.' intact)
sentences = sent_tokenize(text)
print('SENTENCES:')
for i, s in enumerate(sentences, 1):
    print(f' {i}. {s}')

# Word tokenization - splits into words + punctuation
words = word_tokenize(text)
print('\nWORDS:')
for i, s in enumerate(words, 1):
    print(f' {i}. {s}')

print(f'\n{len(sentences)} sentences, {len(words)} word-tokens')

SENTENCES:
 1. Dr. Smith isn't happy.
 2. NLP is fun, and it's powerful!
 3. Visit openai.com today.

WORDS:
 1. Dr.
 2. Smith
 3. is
 4. n't
 5. happy
 6. .
 7. NLP
 8. is
 9. fun
 10. ,
 11. and
 12. it
 13. 's
 14. powerful
 15. !
 16. Visit
 17. openai.com
 18. today
 19. .

3 sentences, 19 word-tokens


# 2. Stemming

**Idea:** Reduce a word to its root by chopping off endings. `running`, `runs`, `ran`(no) -> `run`. It uses crude rules, so the output is not always a real word (`studies` -> `studi`)

**Trade-off:** Vey fast, but sloppy. Good enough for search engines and rough text matching

The **Porter stemmer** is the classic, **Snowball** is a slightly improved version.

In [3]:
from nltk.stem import PorterStemmer, SnowballStemmer

porter = PorterStemmer()
snowball = SnowballStemmer('english')

words = ['running', 'runs', 'runner', 'easily', 'studies', 'studying', 'happily', 'connection']

print(f"{'WORD':<12}{'PORTER':<12}{'SNOWBALL':<12}")

print('-' * 36)

for w in words:
    print(f"{w:<12}{porter.stem(w):<12}{snowball.stem(w):<12}")

WORD        PORTER      SNOWBALL    
------------------------------------
running     run         run         
runs        run         run         
runner      runner      runner      
easily      easili      easili      
studies     studi       studi       
studying    studi       studi       
happily     happili     happili     
connection  connect     connect     


# 3. Lemmatization

**Idea:** Like stemming, but smart. It uses a dictionary (WordNet) and the word's part-of-speech to return the real base word (the lemma)

`studies` -> `study`, `better` -> `good`, `mice` -> `mouse`

**Trade-off:** Slower and needs a POS hint, but the output is always a valid word. Preferred when accuracy matters.

**Key Gotcha:** By default the lemmatizer assumes every word is a noun. Give it the correct POS tag(`v` for verb, `a` for adjective) and results improve dramatiaclly.

In [4]:
from nltk.stem import WordNetLemmatizer

lemma = WordNetLemmatizer()

print('WITHOUT POS (treated as noun):')
for w in ['running', 'studies', 'better', 'mice', 'was']:
    print(f' {w:<10} -> {lemma.lemmatize(w)}')

print('\nWITH correct POS:')
print(f" running (verb) -> {lemma.lemmatize('running', pos='v')}")
print(f" studies (verb) -> {lemma.lemmatize('studies', pos='v')}")
print(f" better  (adj)  -> {lemma.lemmatize('better', pos='a')}")
print(f" was     (verb) -> {lemma.lemmatize('was', pos='v')}")

print('\nStemming vs Lemmatization on "studies":')

print(f" Porter stem -> {porter.stem('studies')} (not a real word)")
print(f" Lemma (word) -> {lemma.lemmatize('studies', pos='v')}")

WITHOUT POS (treated as noun):
 running    -> running
 studies    -> study
 better     -> better
 mice       -> mouse
 was        -> wa

WITH correct POS:
 running (verb) -> run
 studies (verb) -> study
 better  (adj)  -> good
 was     (verb) -> be

Stemming vs Lemmatization on "studies":
 Porter stem -> studi (not a real word)
 Lemma (word) -> study


# 4. Stopword Removal

**Idea:** Words like `the`, `is`, `at`, `a` appear everywhere and carry little meaning for tasks like classification. Removing them shrinks the data and focuses on content words.

**caution:** Not always safe! For sentiment, `not` matters a lot (`not good` != `good`). Remove stopwords only when they don't hurt your task.

In [5]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

print(f'NLTK has {len(stop_words)} English stopwords. A few:')
print(' ', sorted(list(stop_words))[:20])

sentence = "This is a simple example showing how we can remove the stopwords from a sentence"
tokens = word_tokenize(sentence.lower())

filtered = [w for w in tokens if w not in stop_words]

print(f'\nOriginal ({len(tokens)} tokens) : {tokens}')
print(f'Filterede ({len(filtered)} tokens) : {filtered}')

NLTK has 198 English stopwords. A few:
  ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']

Original (15 tokens) : ['this', 'is', 'a', 'simple', 'example', 'showing', 'how', 'we', 'can', 'remove', 'the', 'stopwords', 'from', 'a', 'sentence']
Filterede (6 tokens) : ['simple', 'example', 'showing', 'remove', 'stopwords', 'sentence']


# Putting steps 1-4 together: a reusable `clean_text` function

This is the standard **preprocessing pipeline** you'll reuse in the project:

lowercase -> tokenize -> remove non-alphabetic -> remove stopwords -> lemmatize

In [6]:
import re 

def clean_text(text):
    """FULL preprocessing pipeline: return a clean, space-joined string"""
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemma.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

sample = "The cats were running quickly across 3 beautiful gardens!!!"
print('Raw:    ', sample)
print('Cleaned:', clean_text(sample))


Raw:     The cats were running quickly across 3 beautiful gardens!!!
Cleaned: cat running quickly across beautiful garden


# 5. Bag of Words (BoW)

**Idea:** ML models need numbers, not words. BoW builds a vocabulary of all words, the represents each document as a count vector - how many times each vocab word appears. Word order is thrown away (hence "bag").

**Example:** Vocabulary = `[dog, cat, runs]`. The sentence "dog runs" -> `[1, 0, 1]`.

In scikit-learn this is `CountVectorizer`

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

import pandas as pd

corpus = [
    'the cat sat on the mat', 
    'the dog sat on the log',
    'cats and dogs are friends',
] 

vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(corpus)

# show as a readable table : rows = documents, columns = vocabulary words
df = pd.DataFrame(bow.toarray(), columns = vectorizer.get_feature_names_out())
df.index = [f'doc{i+1}' for i in range(len(corpus))]
print('Vocabulary:', list(vectorizer.get_feature_names_out()))
print('\nBag-of-Words count Matrix')
df

Vocabulary: ['and', 'are', 'cat', 'cats', 'dog', 'dogs', 'friends', 'log', 'mat', 'on', 'sat', 'the']

Bag-of-Words count Matrix


,and,are,cat,cats,dog,dogs,friends,log,mat,on,sat,the
doc1,0,0,1,0,0,0,0,0,1,1,1,2
doc2,0,0,0,0,1,0,0,1,0,1,1,2
doc3,1,1,0,1,0,1,1,0,0,0,0,0


# 6. TF-IDF (Term Frequency - Inverse Document Frequency)

**Problems with BoW:** common words(like `the`) get high counts but tell us nothing distinctive 

**TF-IDF fixes this** by weighing each words: 
$$ \text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D) $$
1. **Term Frequency (TF):** Measures how frequently a term occurs in a document.
   $$ \text{TF}(t, d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}} $$

2. **Inverse Document Frequency (IDF):** Measures how important a term is across the entire corpus.
   $$ \text{IDF}(t, D) = \log \left( \frac{N}{1 + |\{d \in D : t \in d\}|} + 1 \right) $$

- A word frequent in one document but rare overall -> high score (distinctive!)
- A word in every document (like `the`) -> low score (useless)

TF-IDF almost always beats raw BoW for the text classification



**1. Term Frequency (TF)**
$$TF(t, d) = \frac{\text{Count of } t \text{ in } d}{\text{Total words in } d}$$

**2. Inverse Document Frequency (IDF)**
$$IDF(t, D) = \log \left( \frac{N}{|\{d \in D : t \in d\}|} \right)$$

**3. TF-IDF Score**
$$TF\text{-}IDF(t, d, D) = TF(t, d) \times IDF(t, D)$$


In [10]:
from sklearn.feature_extraction.text import TfidfTransformer

# Use the bag-of-words matrix, not the raw corpus list.
# TfidfTransformer expects a document-term matrix such as the output of CountVectorizer.
tfidf = TfidfTransformer()
matrix = tfidf.fit_transform(bow)

df_tfidf = pd.DataFrame(matrix.toarray(),
                        columns=vectorizer.get_feature_names_out()).round(2)

df_tfidf.index = [f'doc{i+1}' for i in range(len(corpus))]
print('TF-IDF matrix (higher = more distinctive to that document):')
df_tfidf

TF-IDF matrix (higher = more distinctive to that document):


,and,are,cat,cats,dog,dogs,friends,log,mat,on,sat,the
doc1,0.00,0.00,0.43,0.00,0.00,0.00,0.00,0.00,0.43,0.33,0.33,0.65
doc2,0.00,0.00,0.00,0.00,0.43,0.00,0.00,0.43,0.00,0.33,0.33,0.65
doc3,0.45,0.45,0.00,0.45,0.00,0.45,0.45,0.00,0.00,0.00,0.00,0.00


# 7. Word Embeddings - Word2Vec & GloVe

**The big limitation of BoW/TF-IDF** they treat `king` and `queen` as totally unrelated (just different columns). They capture counts, not meaning.

**Word embeddings** map each word to a dense vector (eg. 100 numbers) such that words with similar meaning have similar vectors. This is learned from context - "words that appear in similar surroundings have similar meaning"

The famous result: `vector('king') - vector('man') + vector('womam') approx equal to vector('queen')`

- Word2Vec(Google) - Trains on your text using a neural net (CBOW/Skip-gram)
- GloVE(Stanford) - pre-trained on huge corpora; you just download and look up vectors.

Below we train a tiny Word2Vec on a toy corpus with gensim

In [14]:
from gensim.models import Word2Vec

sentences = [
    ['king', 'is', 'a', 'strong', 'man'],
    ['queen', 'is', 'a', 'wise', 'woman'],
    ['boy', 'will', 'be', 'a', 'man'],
    ['girl', 'will', 'be', 'a', 'woman'],
    ['prince', 'is', 'a', 'young', 'king'],
    ['princess', 'is', 'a', 'young', 'queen'],
    ['man', 'is', 'strong'],
    ['woman', 'is', 'wise'],
    ['prince', 'is', 'a', 'boy', 'who', 'will', 'be', 'king'],
    ['princess', 'is', 'a', 'girl', 'who', 'will', 'be', 'queen'],
]

model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, workers=1, seed=42, epochs=200)

print("Vector for 'king' (first 10 out of 50 dims)")
print(model.wv['king'][:10].round(3))

print("\n Words most similar to 'king':")
for word, score in model.wv.most_similar('king', topn=3):
    print(f' {word:<10} {score:.3f}')

print('\nSimilarity(king, queen) =', round(model.wv.similarity('king', 'queen'), 3))
print('\nSimilarity(king, girl) =', round(model.wv.similarity('king', 'girl'), 3))

Vector for 'king' (first 10 out of 50 dims)
[ 0.004 -0.017 -0.008 -0.005  0.022  0.021 -0.017  0.018 -0.019  0.01 ]

 Words most similar to 'king':
 be         0.350
 will       0.252
 who        0.240

Similarity(king, queen) = 0.206

Similarity(king, girl) = 0.126


# 8. PartofSpeech (POS) Tagging

**Idea:** 

In [15]:
from nltk import pos_tag

sentence = "The quick brown fox jumps over the lazy dog"
tokens = word_tokenize(sentence)
tagged = pos_tag(tokens)

print('POS tags:')
for word, tag in tagged:
    print(f'{word:<10} {tag}')

print('\nCommon tags: NN=noun, NNP=proper noun, VB=verb, JJ=adjective, DT=determiner, IN=preposition')

POS tags:
The        DT
quick      JJ
brown      NN
fox        NN
jumps      VBZ
over       IN
the        DT
lazy       JJ
dog        NN

Common tags: NN=noun, NNP=proper noun, VB=verb, JJ=adjective, DT=determiner, IN=preposition
